In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Tuple, Optional, List, Set
import csv
import json
import time
import warnings

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ─── Paths ──────────────────────────────────────────────────────────────────
BASE = Path(".")
GAMES_LIVE2  = BASE / "data" / "games_live2"
PLAYERS_LIVE = BASE / "data" / "players_live"
PREDICTIONS  = BASE / "data" / "games_predictions.csv"
EVENTS_JSON  = BASE / "data" / "game_events.json"
OUTPUT_PATH  = BASE / "data" / "sample_pbp_train.csv"

# ─── Sample-weight constants (tunable) ──────────────────────────────────────
TOTAL_GAME_SEC = 48 * 60          # 2880  (ignore OT — clamp to 0)
W_A, W_P       = 2.0, 2.0        # time-urgency factor
W_B, W_S       = 2.0, 6.0        # closeness factor
MIN_W, MAX_W   = 1.0, 10.0       # clip bounds

# ─── Down-sampling ──────────────────────────────────────────────────────────
CLUTCH_SEC      = 300             # keep ALL events in last 5 min
SAMPLE_INTERVAL = 30              # ≤ 1 snapshot per 30 s before that

# ─── Rolling-feature window ────────────────────────────────────────────────
ROLL_WINDOW_SEC = 120             # 2-minute rolling window

# ─── Event-type ID sets ────────────────────────────────────────────────────
FT_IDS:    Set[int] = {97, 98, 99, 100, 101, 102, 103, 104, 105, 106,
                        107, 108, 157, 165, 166}
# SHOT_IDS includes ALL shot-like events (field goals + free throws)
ALL_SHOT_IDS: Set[int] = set(range(91, 154)) | {282}
# FG_IDS = field-goal attempts only (exclude free throws for FGA counting)
FG_IDS:    Set[int] = ALL_SHOT_IDS - FT_IDS
TOV_IDS:   Set[int] = set(range(62, 79)) | {84, 86, 87, 88, 89, 90, 478, 592}
FOUL_IDS:  Set[int] = {22, 24, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41,
                        42, 43, 44, 45, 47, 48, 257}
REB_IDS:   Set[int] = {155, 156}
DEF_REB, OFF_REB = 155, 156

STATE_CHANGE: Set[int] = ALL_SHOT_IDS | FT_IDS | TOV_IDS | FOUL_IDS | REB_IDS
EXCLUDE_IDS:  Set[int] = {0, 402, 411, 412}  # non-game meta-events

# ─── PBP columns we need ───────────────────────────────────────────────────
PBP_USECOLS = [
    "sequence_number", "type_id", "type_text",
    "away_score", "home_score", "period_number",
    "clock_minutes", "clock_seconds",
    "scoring_play", "score_value",
    "end_game_seconds_remaining", "end_quarter_seconds_remaining",
    "season", "team_id",
    "away_team_id", "home_team_id", "game_date",
]

# ─── Load event map (debugging reference) ──────────────────────────────────
with open(EVENTS_JSON) as _f:
    EVENT_MAP: Dict[str, str] = json.load(_f)

print("Constants loaded")
print(f"  State-changing event types: {len(STATE_CHANGE)}")
print(f"  Paths OK: GAMES_LIVE2={GAMES_LIVE2.exists()}, "
      f"PREDICTIONS={PREDICTIONS.exists()}")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Constants loaded
  State-changing event types: 113
  Paths OK: GAMES_LIVE2=True, PREDICTIONS=True


In [2]:
# ─── File-discovery helpers ─────────────────────────────────────────────────

def season_from_date(dt: str) -> int:
    """'YYYY-MM-DD' -> NBA season-folder int (e.g. 2008 for the 2007-08 season).

    Convention: if month >= 7 (July+), the game belongs to the *next* season.
    """
    y, m = int(dt[:4]), int(dt[5:7])
    return y + 1 if m >= 7 else y


def build_pbp_index(root: Path) -> Dict[Tuple[str, str, str], List[Path]]:
    """Build filename-only index: (season_str, away_abbrev, home_abbrev) -> [paths].

    No file I/O — purely from directory listings.
    """
    idx: Dict[Tuple[str, str, str], List[Path]] = {}
    for sdir in sorted(root.iterdir()):
        if not sdir.is_dir():
            continue
        s = sdir.name
        for f in sdir.iterdir():
            if f.suffix != ".csv":
                continue
            parts = f.stem.split("_")
            if len(parts) < 3:
                continue
            idx.setdefault((s, parts[1], parts[2]), []).append(f)
    return idx


def resolve_pbp(
    dt: str, away: str, home: str,
    idx: Dict[Tuple[str, str, str], List[Path]],
) -> Optional[Path]:
    """Return the PBP file for a game, or None if not found.

    Disambiguates by reading the first row's game_date when multiple candidates
    exist for the same (season, away, home) key.
    """
    season = str(season_from_date(dt))
    cands = idx.get((season, away, home), [])
    if not cands:
        return None
    if len(cands) == 1:
        return cands[0]
    # Multiple candidates (same teams, same season) — read first row to check date
    for c in cands:
        try:
            with open(c) as fh:
                row = next(csv.DictReader(fh))
                if row.get("game_date", "").split("T")[0] == dt:
                    return c
        except (StopIteration, KeyError):
            continue
    return None


def resolve_players(gid: str, away: str, home: str) -> Optional[Path]:
    """Return players_live CSV path, or None.

    Players_live files use the NBA-stats game ID with a '00' prefix:
    00{game_id}_{away}_{home}.csv
    """
    p = PLAYERS_LIVE / f"00{gid}_{away}_{home}.csv"
    return p if p.exists() else None


# ─── Build the PBP index (one-time cost) ───────────────────────────────────
t0 = time.time()
PBP_IDX = build_pbp_index(GAMES_LIVE2)
n_files = sum(len(v) for v in PBP_IDX.values())
n_seasons = len({k[0] for k in PBP_IDX})
print(f"PBP index built: {n_files:,} files across {n_seasons} seasons "
      f"({time.time() - t0:.1f}s)")

PBP index built: 30,516 files across 25 seasons (0.3s)


In [3]:
# ─── Feature-computation helpers ────────────────────────────────────────────

def vec_sec_remaining_game(pbp: pd.DataFrame) -> np.ndarray:
    """Vectorised game-seconds remaining.

    Primary: end_game_seconds_remaining column.
    Fallback (per-row): (4 - period) * 600 + clock_minutes * 60 + clock_seconds.
    The data pipeline uses 600 s/period for remaining periods (not 720),
    so the fallback mirrors that convention.
    """
    sec = pd.to_numeric(pbp.get("end_game_seconds_remaining"), errors="coerce").values.astype(float)
    mask = np.isnan(sec)
    if mask.any():
        p  = pd.to_numeric(pbp.loc[mask, "period_number"], errors="coerce").fillna(4).astype(int).values
        cm = pd.to_numeric(pbp.loc[mask, "clock_minutes"],  errors="coerce").fillna(0).astype(float).values
        cs = pd.to_numeric(pbp.loc[mask, "clock_seconds"],  errors="coerce").fillna(0).astype(float).values
        sec[mask] = (4 - p) * 600.0 + cm * 60.0 + cs
    return np.maximum(sec, 0.0)


def vec_sec_remaining_period(pbp: pd.DataFrame) -> np.ndarray:
    """Vectorised period-seconds remaining."""
    sec = pd.to_numeric(pbp.get("end_quarter_seconds_remaining"), errors="coerce").values.astype(float)
    mask = np.isnan(sec)
    if mask.any():
        cm = pd.to_numeric(pbp.loc[mask, "clock_minutes"], errors="coerce").fillna(0).astype(float).values
        cs = pd.to_numeric(pbp.loc[mask, "clock_seconds"], errors="coerce").fillna(0).astype(float).values
        sec[mask] = cm * 60.0 + cs
    return np.maximum(sec, 0.0)


def rolling_sum_time(
    vals: np.ndarray,
    sec_rem: np.ndarray,
    window: float,
) -> np.ndarray:
    """O(n) rolling sum over a time-based window.

    For event i (sorted chronologically — sec_rem approximately non-increasing),
    sums vals[j] for all j <= i where sec_rem[j] - sec_rem[i] <= window.
    """
    n = len(vals)
    out = np.empty(n, dtype=np.float64)
    running = 0.0
    left = 0
    for i in range(n):
        running += vals[i]
        while left < i and sec_rem[left] - sec_rem[i] > window:
            running -= vals[left]
            left += 1
        out[i] = running
    return out


def sample_weight_vec(sec: np.ndarray, sdiff: np.ndarray) -> np.ndarray:
    """Vectorised leverage-weighted sample weight.

    time_factor  = 1 + a * (1 - sec/total)^p     (late game = higher)
    close_factor = 1 + b * exp(-|diff|/s)         (close game = higher)
    """
    sc = np.clip(sec, 0.0, TOTAL_GAME_SEC).astype(float)
    tf = 1.0 + W_A * (1.0 - sc / TOTAL_GAME_SEC) ** W_P
    cf = 1.0 + W_B * np.exp(-np.abs(sdiff.astype(float)) / W_S)
    return np.clip(tf * cf, MIN_W, MAX_W)


print("Feature helpers defined")

Feature helpers defined


In [4]:
# ─── Single-game processor ──────────────────────────────────────────────────

def process_game(
    game_id: str,
    game_date: str,
    home_team: str,
    away_team: str,
    prior_home_wp: float,
    pbp_path: Path,
) -> pd.DataFrame:
    """Read one game's PBP CSV, compute all features, downsample, return training rows.

    Raises on data-integrity errors so the caller can log and continue.
    """

    # ── 1. Read & sort ──────────────────────────────────────────────────
    try:
        pbp = pd.read_csv(pbp_path, usecols=PBP_USECOLS)
    except ValueError:
        # Some files may lack optional columns — read everything
        pbp = pd.read_csv(pbp_path)

    assert len(pbp) > 0, f"Empty PBP file: {pbp_path}"
    pbp.sort_values("sequence_number", inplace=True)
    pbp.reset_index(drop=True, inplace=True)
    n = len(pbp)

    # ── 2. Coerce numeric types ─────────────────────────────────────────
    NUM_COLS = [
        "away_score", "home_score", "type_id", "score_value",
        "period_number", "clock_minutes", "clock_seconds",
        "end_game_seconds_remaining", "end_quarter_seconds_remaining",
        "team_id", "away_team_id", "home_team_id",
    ]
    for c in NUM_COLS:
        if c in pbp.columns:
            pbp[c] = pd.to_numeric(pbp[c], errors="coerce")

    pbp["score_value"] = pbp["score_value"].fillna(0).astype(int)
    tids = pbp["type_id"].fillna(-1).astype(int).values

    # ── 3. Team IDs ─────────────────────────────────────────────────────
    htid_col = pbp["home_team_id"].dropna()
    atid_col = pbp["away_team_id"].dropna()
    assert not htid_col.empty, f"No home_team_id for game {game_id} ({pbp_path})"
    assert not atid_col.empty, f"No away_team_id for game {game_id} ({pbp_path})"
    HID = int(htid_col.iloc[0])
    AID = int(atid_col.iloc[0])
    season = int(pbp["season"].dropna().iloc[0]) if "season" in pbp.columns else None

    # ── 4. Final outcome ────────────────────────────────────────────────
    final_h = pbp["home_score"].max()
    final_a = pbp["away_score"].max()
    assert pd.notna(final_h) and pd.notna(final_a), \
        f"Cannot determine final score for game {game_id}"
    home_win = int(final_h > final_a)

    # ── 5. Vectorised base arrays ───────────────────────────────────────
    tmids  = pbp["team_id"].values                        # float, NaN possible
    ttexts = pbp["type_text"].fillna("").astype(str).values

    sec_g = vec_sec_remaining_game(pbp)
    sec_p = vec_sec_remaining_period(pbp)

    h_sc = pbp["home_score"].ffill().fillna(0).astype(int).values
    a_sc = pbp["away_score"].ffill().fillna(0).astype(int).values
    s_diff = h_sc - a_sc

    # is_home_event  (1.0 = home, 0.0 = away, NaN = unknown)
    ihe = np.where(tmids == HID, 1.0,
                   np.where(tmids == AID, 0.0, np.nan))

    # scoring_play flag — handles string "True"/"False" and bool
    sp_raw = pbp["scoring_play"].astype(str).str.strip().str.lower()
    is_sc = sp_raw.isin(["true", "1"]).astype(int).values
    sv = pbp["score_value"].values

    # ── 6. Event-type boolean arrays (vectorised with np.isin) ──────────
    _fg    = np.isin(tids, list(FG_IDS))           # field-goal attempts
    _ft    = np.isin(tids, list(FT_IDS))            # free-throw attempts
    _any_shot = np.isin(tids, list(ALL_SHOT_IDS))   # any shot (FG + FT)
    _tov   = np.isin(tids, list(TOV_IDS))
    _dreb  = (tids == DEF_REB)
    _oreb  = (tids == OFF_REB)

    h_fg   = (ihe == 1) & _fg;     a_fg   = (ihe == 0) & _fg
    h_ft   = (ihe == 1) & _ft;     a_ft   = (ihe == 0) & _ft
    h_tov  = (ihe == 1) & _tov;    a_tov  = (ihe == 0) & _tov
    h_oreb = (ihe == 1) & _oreb;   a_oreb = (ihe == 0) & _oreb
    h_dreb = (ihe == 1) & _dreb;   a_dreb = (ihe == 0) & _dreb

    # ── 7. Possession inference (sequential — must iterate) ─────────────
    poss = np.full(n, np.nan)
    cur: float = np.nan
    for i in range(n):
        t = tids[i]
        tm = tmids[i]
        if t != -1 and not np.isnan(tm):
            tm_int = int(tm)
            if t in ALL_SHOT_IDS or t in FT_IDS:
                cur = float(tm_int)
            elif t in TOV_IDS or "Turnover" in ttexts[i]:
                cur = float(tm_int)
            elif t == DEF_REB or t == OFF_REB:
                cur = float(tm_int)
        poss[i] = cur

    ihp = np.where(np.isnan(poss), np.nan,
                   np.where(poss == HID, 1.0, 0.0))

    # ── 8. Rolling 120-s features ───────────────────────────────────────
    h_sc_vals = np.where((ihe == 1) & (is_sc == 1), sv.astype(float), 0.0)
    a_sc_vals = np.where((ihe == 0) & (is_sc == 1), sv.astype(float), 0.0)

    hp120 = rolling_sum_time(h_sc_vals, sec_g, ROLL_WINDOW_SEC)
    ap120 = rolling_sum_time(a_sc_vals, sec_g, ROLL_WINDOW_SEC)
    ht120 = rolling_sum_time(h_tov.astype(float), sec_g, ROLL_WINDOW_SEC)
    at120 = rolling_sum_time(a_tov.astype(float), sec_g, ROLL_WINDOW_SEC)

    # ── 9. Cumulative stats ─────────────────────────────────────────────
    h_fga = np.cumsum(h_fg.astype(int))
    a_fga = np.cumsum(a_fg.astype(int))
    h_fta = np.cumsum(h_ft.astype(int))
    a_fta = np.cumsum(a_ft.astype(int))
    h_or  = np.cumsum(h_oreb.astype(int))
    a_or  = np.cumsum(a_oreb.astype(int))
    h_dr  = np.cumsum(h_dreb.astype(int))
    a_dr  = np.cumsum(a_dreb.astype(int))

    # ── 10. Assemble full feature matrix ────────────────────────────────
    feat = pd.DataFrame({
        "game_id":              game_id,
        "season":               season,
        "game_date":            game_date,
        "play_index":           pbp["sequence_number"].values,
        "prior_home_wp":        prior_home_wp,
        "home_score":           h_sc,
        "away_score":           a_sc,
        "score_diff":           s_diff,
        "period_number":        pbp["period_number"].values,
        "sec_remaining_game":   sec_g,
        "sec_remaining_period": sec_p,
        "is_home_possession":   ihp,
        "event_type_id":        tids,
        "is_scoring_play":      is_sc,
        "score_value":          sv,
        "home_points_last_120": hp120,
        "away_points_last_120": ap120,
        "home_tov_last_120":    ht120,
        "away_tov_last_120":    at120,
        "home_fga_to_date":     h_fga,
        "away_fga_to_date":     a_fga,
        "home_fta_to_date":     h_fta,
        "away_fta_to_date":     a_fta,
        "home_oreb_to_date":    h_or,
        "away_oreb_to_date":    a_or,
        "home_dreb_to_date":    h_dr,
        "away_dreb_to_date":    a_dr,
        "home_win_final":       home_win,
    })

    # ── 11. Exclude meta-events ─────────────────────────────────────────
    feat = feat[~feat["event_type_id"].isin(EXCLUDE_IDS)].copy()

    # ── 12. Downsample ──────────────────────────────────────────────────
    is_clutch = feat["sec_remaining_game"] <= CLUTCH_SEC
    late  = feat[is_clutch]                                      # keep everything
    early = feat[~is_clutch].copy()

    # Before clutch zone: only state-changing events, ≤1 per 30-s bin
    early = early[early["event_type_id"].isin(STATE_CHANGE)].copy()
    if not early.empty:
        early["_bin"] = (early["sec_remaining_game"] // SAMPLE_INTERVAL).astype(int)
        early = early.groupby("_bin", sort=False).last().reset_index(drop=True)

    out = pd.concat([early, late], ignore_index=True)
    out.sort_values("play_index", inplace=True)
    out.reset_index(drop=True, inplace=True)
    if "_bin" in out.columns:
        out.drop(columns=["_bin"], inplace=True)

    # ── 13. Sample weights ──────────────────────────────────────────────
    out["sample_weight"] = sample_weight_vec(
        out["sec_remaining_game"].values,
        out["score_diff"].values,
    )

    return out


print("process_game defined")

process_game defined


In [8]:
# ─── Main pipeline ──────────────────────────────────────────────────────────

predictions = pd.read_csv(PREDICTIONS, dtype={"game_id": str})
print(f"Loaded {len(predictions):,} prediction entries")

# ── TEST MODE: restrict to a single season for fast iteration ───────────
TEST_SEASON: Optional[int] = None   # set to None to process ALL seasons
if TEST_SEASON is not None:
    dates = pd.to_datetime(predictions["game_date"])
    predictions["_season"] = dates.dt.year.where(dates.dt.month < 7, dates.dt.year + 1)
    predictions = predictions[predictions["_season"] == TEST_SEASON].copy()
    predictions.drop(columns=["_season"], inplace=True)
    print(f"  TEST MODE: filtered to season {TEST_SEASON} → {len(predictions):,} games")

print(f"  Date range: {predictions['game_date'].min()} → {predictions['game_date'].max()}")

results: List[pd.DataFrame] = []
skipped_no_pbp = 0
skipped_error  = 0
errors_log: List[str] = []

t0 = time.time()
total = len(predictions)

for i, row in predictions.iterrows():
    gid  = str(row["game_id"])
    gdt  = str(row["game_date"])
    ht   = str(row["home_team"])
    at   = str(row["away_team"])
    pwp  = float(row["pred_prob"])

    # Resolve PBP file
    pbp_path = resolve_pbp(gdt, at, ht, PBP_IDX)
    if pbp_path is None:
        skipped_no_pbp += 1
        continue

    try:
        df = process_game(gid, gdt, ht, at, pwp, pbp_path)
        results.append(df)
    except Exception as e:
        skipped_error += 1
        msg = f"ERROR game {gid} ({gdt} {at}@{ht}): {e}"
        errors_log.append(msg)
        if skipped_error <= 20:
            print(msg)

    # Progress every 2000 games
    done = len(results) + skipped_no_pbp + skipped_error
    if done % 2000 == 0:
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        print(f"  {done:>6,} / {total:,}  |  "
              f"{len(results):,} ok, {skipped_no_pbp} no-pbp, {skipped_error} err  |  "
              f"{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

elapsed = time.time() - t0
print(f"\nPipeline complete in {elapsed:.0f}s")
print(f"  Processed:        {len(results):,}")
print(f"  Skipped (no PBP): {skipped_no_pbp:,}")
print(f"  Skipped (error):  {skipped_error:,}")

if errors_log:
    print(f"\n--- First {min(20, len(errors_log))} errors ---")
    for e in errors_log[:20]:
        print(f"  {e}")

Loaded 28,429 prediction entries
  Date range: 2000-11-07 → 2025-04-13
   2,000 / 28,429  |  426 ok, 1574 no-pbp, 0 err  |  7s elapsed, ~99s remaining
   4,000 / 28,429  |  2,321 ok, 1679 no-pbp, 0 err  |  41s elapsed, ~252s remaining
   6,000 / 28,429  |  4,312 ok, 1688 no-pbp, 0 err  |  76s elapsed, ~283s remaining
   8,000 / 28,429  |  6,300 ok, 1700 no-pbp, 0 err  |  111s elapsed, ~284s remaining
  10,000 / 28,429  |  8,296 ok, 1704 no-pbp, 0 err  |  147s elapsed, ~270s remaining
  12,000 / 28,429  |  10,292 ok, 1708 no-pbp, 0 err  |  183s elapsed, ~251s remaining
  14,000 / 28,429  |  12,292 ok, 1708 no-pbp, 0 err  |  220s elapsed, ~227s remaining
  16,000 / 28,429  |  14,287 ok, 1713 no-pbp, 0 err  |  259s elapsed, ~201s remaining
  18,000 / 28,429  |  16,286 ok, 1714 no-pbp, 0 err  |  296s elapsed, ~171s remaining
  20,000 / 28,429  |  18,285 ok, 1715 no-pbp, 0 err  |  332s elapsed, ~140s remaining
  22,000 / 28,429  |  20,284 ok, 1716 no-pbp, 0 err  |  370s elapsed, ~108s remai

In [10]:
# ─── Concatenate, verify, and save ──────────────────────────────────────────

if not results:
    raise RuntimeError("No games were processed — nothing to save!")

output = pd.concat(results, ignore_index=True)

print(f"Output shape: {output.shape[0]:,} rows x {output.shape[1]} columns")
print(f"Columns: {list(output.columns)}\n")

print("Label distribution (home_win_final):")
print(output["home_win_final"].value_counts().to_string())
print(f"  Home win rate: {output['home_win_final'].mean():.4f}\n")

print("Sample weight stats:")
print(output["sample_weight"].describe().to_string())
print()

print(f"sec_remaining_game range: "
      f"[{output['sec_remaining_game'].min():.0f}, "
      f"{output['sec_remaining_game'].max():.0f}]")
print(f"Unique games: {output['game_id'].nunique():,}")
print(f"Rows per game (median): {output.groupby('game_id').size().median():.0f}")
print(f"Seasons covered: {sorted(output['season'].dropna().unique().astype(int))}\n")

# ── Sanity checks ──────────────────────────────────────────────────────
assert output["home_win_final"].isin([0, 1]).all(), "home_win_final has values outside {0,1}"
assert output["sample_weight"].between(MIN_W, MAX_W).all(), "sample_weight out of bounds"
assert output["sec_remaining_game"].ge(0).all(), "negative sec_remaining_game found"
print("Sanity checks passed\n")

# ── Save ────────────────────────────────────────────────────────────────
output.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(output):,} rows to {OUTPUT_PATH}")

# ── Preview ─────────────────────────────────────────────────────────────
output.head(5)

Output shape: 3,542,891 rows x 29 columns
Columns: ['game_id', 'season', 'game_date', 'play_index', 'prior_home_wp', 'home_score', 'away_score', 'score_diff', 'period_number', 'sec_remaining_game', 'sec_remaining_period', 'is_home_possession', 'event_type_id', 'is_scoring_play', 'score_value', 'home_points_last_120', 'away_points_last_120', 'home_tov_last_120', 'away_tov_last_120', 'home_fga_to_date', 'away_fga_to_date', 'home_fta_to_date', 'away_fta_to_date', 'home_oreb_to_date', 'away_oreb_to_date', 'home_dreb_to_date', 'away_dreb_to_date', 'home_win_final', 'sample_weight']

Label distribution (home_win_final):
home_win_final
1    2055830
0    1487061
  Home win rate: 0.5803

Sample weight stats:
count    3.542891e+06
mean     3.687723e+00
std      1.679680e+00
min      1.068252e+00
25%      2.492943e+00
50%      3.170887e+00
75%      4.552029e+00
max      9.000000e+00

sec_remaining_game range: [0, 2880]
Unique games: 26,611
Rows per game (median): 130
Seasons covered: [2002, 2003,

,game_id,season,game_date,play_index,prior_home_wp,home_score,away_score,score_diff,period_number,sec_remaining_game,...,home_fga_to_date,away_fga_to_date,home_fta_to_date,away_fta_to_date,home_oreb_to_date,away_oreb_to_date,home_dreb_to_date,away_dreb_to_date,home_win_final,sample_weight
0,211107018,2002,2001-11-07,3,0.128044,3,0,3,1,2520.0,...,1,0,0,0,0,0,0,0,1,2.282219
1,211107018,2002,2001-11-07,4,0.128044,3,2,1,1,2501.0,...,1,1,0,0,0,0,0,0,1,2.786236
2,211107018,2002,2001-11-07,6,0.128044,6,2,4,1,2463.0,...,2,2,0,0,0,0,0,0,1,2.111818
3,211107018,2002,2001-11-07,9,0.128044,8,2,6,1,2436.0,...,3,3,0,0,0,0,1,0,1,1.818268
4,211107018,2002,2001-11-07,13,0.128044,10,4,6,1,2415.0,...,4,4,0,0,0,0,1,1,1,1.826257


---
# Part 2 — Model Training, Calibration & Evaluation
---

## 1. Imports & Config

In [ ]:
# ─── 1. Imports & Config ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for clean PNG export
import matplotlib.pyplot as plt
import joblib
import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    RUN_XGB = True
    print("xgboost available")
except ImportError:
    RUN_XGB = False
    print("xgboost NOT installed — skipping XGB model")

# ─── Global config ──────────────────────────────────────────────────────────
DATA_PATH   = "data/sample_pbp_train.csv"
OUT_DIR     = "outputs"
RANDOM_SEED = 42
TOTAL_GAME_SECONDS = 2880       # 48 × 60, used for derived features

print(f"Config: DATA_PATH={DATA_PATH}, OUT_DIR={OUT_DIR}, SEED={RANDOM_SEED}")

## 2. Utility Functions

In [ ]:
# ─── 2. Utility Functions ───────────────────────────────────────────────────

def set_seeds(seed: int = RANDOM_SEED) -> None:
    """Set random seeds for reproducibility."""
    np.random.seed(seed)
    print(f"Random seed set to {seed}")


def ensure_out_dir(path: str = OUT_DIR) -> Path:
    """Create output directory if it does not exist."""
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


# ── Weighted metrics ────────────────────────────────────────────────────────

def weighted_log_loss(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    w: np.ndarray,
    eps: float = 1e-15,
) -> float:
    """Weighted binary log-loss."""
    p = np.clip(y_prob, eps, 1 - eps)
    ll = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    return float(np.average(ll, weights=w))


def weighted_brier(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    w: np.ndarray,
) -> float:
    """Weighted Brier score."""
    return float(np.average((y_prob - y_true) ** 2, weights=w))


def weighted_auc(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    w: np.ndarray,
) -> float:
    """Weighted ROC-AUC via sklearn (uses sample_weight)."""
    return float(roc_auc_score(y_true, y_prob, sample_weight=w))


def expected_calibration_error(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    w: np.ndarray,
    n_bins: int = 15,
) -> float:
    """Weighted Expected Calibration Error (ECE)."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total_w = w.sum()
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if not mask.any():
            continue
        bw = w[mask].sum()
        mean_pred = np.average(y_prob[mask], weights=w[mask])
        mean_true = np.average(y_true[mask], weights=w[mask])
        ece += (bw / total_w) * abs(mean_pred - mean_true)
    return float(ece)


def compute_all_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    w: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Compute full metric suite (weighted + unweighted)."""
    ones = np.ones_like(w)
    m: Dict[str, float] = {}
    for suffix, wt in [("_w", w), ("_uw", ones)]:
        tag = prefix + suffix
        m[f"{tag}_logloss"]  = weighted_log_loss(y_true, y_prob, wt)
        m[f"{tag}_brier"]    = weighted_brier(y_true, y_prob, wt)
        m[f"{tag}_ece"]      = expected_calibration_error(y_true, y_prob, wt)
        m[f"{tag}_auc"]      = weighted_auc(y_true, y_prob, wt)
    return m


# ── Plotting ────────────────────────────────────────────────────────────────

def plot_reliability_diagram(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    w: np.ndarray,
    n_bins: int = 15,
    title: str = "Reliability Diagram",
    save_path: Optional[str] = None,
) -> None:
    """Plot a weighted reliability / calibration diagram."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    mean_pred, mean_true, bin_weight = [], [], []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() < 5:
            continue
        bw = w[mask]
        mean_pred.append(np.average(y_prob[mask], weights=bw))
        mean_true.append(np.average(y_true[mask], weights=bw))
        bin_weight.append(bw.sum())

    mean_pred = np.array(mean_pred)
    mean_true = np.array(mean_true)
    bin_weight = np.array(bin_weight)
    bin_weight = bin_weight / bin_weight.max()   # normalise for bar width

    fig, ax1 = plt.subplots(figsize=(6, 5))
    ax1.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    ax1.plot(mean_pred, mean_true, "o-", color="steelblue", label="Model")
    ax1.set_xlabel("Mean predicted probability")
    ax1.set_ylabel("Fraction of positives (weighted)")
    ax1.set_xlim(-0.02, 1.02)
    ax1.set_ylim(-0.02, 1.02)
    ax1.set_title(title)
    ax1.legend(loc="upper left")

    # histogram of predictions on twin axis
    ax2 = ax1.twinx()
    ax2.bar(
        mean_pred, bin_weight, width=1 / n_bins * 0.8,
        alpha=0.25, color="grey", label="Relative bin weight",
    )
    ax2.set_ylabel("Relative bin weight")
    ax2.set_ylim(0, 2.5)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved plot → {save_path}")
    plt.close(fig)


set_seeds()
out_dir = ensure_out_dir()
print(f"Output directory: {out_dir.resolve()}")

## 3. Load & Clean Data

In [ ]:
# ─── 3. Load & Clean Data ───────────────────────────────────────────────────
t0 = time.time()
raw = pd.read_csv(DATA_PATH, dtype={"game_id": str})
print(f"Loaded {len(raw):,} rows in {time.time()-t0:.1f}s")

# Parse types
raw["game_date"] = pd.to_datetime(raw["game_date"], errors="coerce")
raw["season"] = raw["season"].astype(int)
raw["home_win_final"] = raw["home_win_final"].astype(int)
assert raw["home_win_final"].isin([0, 1]).all(), "Label not in {0,1}"

# Clean sample_weight
raw["sample_weight"] = pd.to_numeric(raw["sample_weight"], errors="coerce")
raw["sample_weight"] = raw["sample_weight"].fillna(1.0).clip(lower=1e-6)

# Fill NaN in is_home_possession with -1 (explicit unknown)
raw["is_home_possession"] = raw["is_home_possession"].fillna(-1.0)

# Sort for reproducibility
raw.sort_values(["game_id", "play_index"], inplace=True)
raw.reset_index(drop=True, inplace=True)

# ── Summary ─────────────────────────────────────────────────────────────
n_games = raw["game_id"].nunique()
seasons = sorted(raw["season"].unique())
label_mean = raw["home_win_final"].mean()
label_wmean = np.average(raw["home_win_final"], weights=raw["sample_weight"])

print(f"  Rows:    {len(raw):,}")
print(f"  Games:   {n_games:,}")
print(f"  Seasons: {seasons[0]}–{seasons[-1]} ({len(seasons)} total)")
print(f"  Label balance (unweighted): {label_mean:.4f}")
print(f"  Label balance (weighted):   {label_wmean:.4f}")
print(f"  sample_weight range: [{raw['sample_weight'].min():.3f}, {raw['sample_weight'].max():.3f}]")

## 4. Train / Val / Test Split (no leakage)

In [ ]:
# ─── 4. Train / Val / Test Split ────────────────────────────────────────────
#
# Temporal split by season to prevent leakage:
#   Train : seasons <= max_season - 2
#   Val   : season == max_season - 1
#   Test  : season == max_season
# Fallback to GroupShuffleSplit if only one season.

max_season = raw["season"].max()
n_seasons  = raw["season"].nunique()

if n_seasons >= 3:
    train_seasons = [s for s in seasons if s <= max_season - 2]
    val_season    = max_season - 1
    test_season   = max_season

    df_train = raw[raw["season"].isin(train_seasons)].copy()
    df_val   = raw[raw["season"] == val_season].copy()
    df_test  = raw[raw["season"] == test_season].copy()
    split_desc = (f"Temporal split — Train: {train_seasons[0]}–{train_seasons[-1]}, "
                  f"Val: {val_season}, Test: {test_season}")
else:
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
    rest_idx, test_idx = next(gss.split(raw, groups=raw["game_id"]))
    rest = raw.iloc[rest_idx]
    df_test = raw.iloc[test_idx].copy()

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=RANDOM_SEED)  # ~15% of total
    train_idx, val_idx = next(gss2.split(rest, groups=rest["game_id"]))
    df_train = rest.iloc[train_idx].copy()
    df_val   = rest.iloc[val_idx].copy()
    split_desc = "GroupShuffleSplit (single-season fallback)"

# ── Verify no game leakage ──────────────────────────────────────────────
train_gids = set(df_train["game_id"])
val_gids   = set(df_val["game_id"])
test_gids  = set(df_test["game_id"])
assert train_gids.isdisjoint(val_gids),  "LEAK: train ∩ val"
assert train_gids.isdisjoint(test_gids), "LEAK: train ∩ test"
assert val_gids.isdisjoint(test_gids),   "LEAK: val ∩ test"

print(f"Split: {split_desc}")
for name, df_ in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    ng = df_["game_id"].nunique()
    ss = sorted(df_["season"].unique())
    print(f"  {name:5s}: {len(df_):>10,} rows, {ng:>5,} games, seasons {ss[0]}–{ss[-1]}")
print("No game-ID leakage across splits")

## 5. Feature Engineering

In [ ]:
# ─── 5. Feature Engineering ─────────────────────────────────────────────────

TARGET_COL = "home_win_final"
WEIGHT_COL = "sample_weight"

# Columns to drop from features
ID_COLS   = ["game_id", "play_index", "game_date"]
META_COLS = [TARGET_COL, WEIGHT_COL, "season"]
DROP_COLS = ID_COLS + META_COLS

# Categorical column(s)
CAT_COLS = ["event_type_id"]

# All remaining columns are numeric features
NUMERIC_COLS: List[str] = []   # populated after first call


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add safe, row-local derived features (in-place for speed)."""
    df = df.copy()
    df["log_sec_game"]   = np.log1p(df["sec_remaining_game"].clip(lower=0))
    df["log_sec_period"] = np.log1p(df["sec_remaining_period"].clip(lower=0))
    df["time_frac"]      = (df["sec_remaining_game"] / TOTAL_GAME_SECONDS).clip(0, 1)
    df["score_time_ix"]  = df["score_diff"] * df["time_frac"]
    return df


def make_features(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """Return (X, y, w) from a split DataFrame.

    X  — feature DataFrame (includes cat + numeric cols, no IDs/label/weight)
    y  — 1-d int array
    w  — 1-d float array
    """
    global NUMERIC_COLS
    df = add_derived_features(df)
    y = df[TARGET_COL].values.astype(int)
    w = df[WEIGHT_COL].values.astype(float)
    X = df.drop(columns=DROP_COLS, errors="ignore")

    # Identify numeric vs categorical
    if not NUMERIC_COLS:
        NUMERIC_COLS = [c for c in X.columns if c not in CAT_COLS]
    return X, y, w


# ── Build feature matrices ──────────────────────────────────────────────
X_train, y_train, w_train = make_features(df_train)
X_val,   y_val,   w_val   = make_features(df_val)
X_test,  y_test,  w_test  = make_features(df_test)

print(f"Feature columns ({len(X_train.columns)}):")
print(f"  Numeric ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"  Categorical: {CAT_COLS}")
print(f"\nTrain X shape: {X_train.shape}")
print(f"Val   X shape: {X_val.shape}")
print(f"Test  X shape: {X_test.shape}")

# ── Preprocessor (ColumnTransformer) for pipeline-based models ──────────
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("num", StandardScaler(), NUMERIC_COLS),
    ],
    remainder="drop",
)
print("\nPreprocessor defined (OneHotEncoder + StandardScaler)")

## 6. Baselines & Models

In [ ]:
# ─── 6. Baselines & Models ──────────────────────────────────────────────────
#
# We store every trained model + its val/test raw predictions in a dict:
#   models_registry[name] = {
#       "model": fitted object (or None for baseline),
#       "val_pred": np.ndarray,
#       "test_pred": np.ndarray,
#   }

models_registry: Dict[str, Dict[str, Any]] = {}

# ── A) Baseline: prior_home_wp ──────────────────────────────────────────
print("=" * 60)
print("A) Baseline — prior_home_wp")
print("=" * 60)
val_prior  = df_val["prior_home_wp"].values.clip(1e-6, 1 - 1e-6)
test_prior = df_test["prior_home_wp"].values.clip(1e-6, 1 - 1e-6)
models_registry["prior_baseline"] = {
    "model": None,
    "val_pred": val_prior,
    "test_pred": test_prior,
}
print(f"  Val logloss (weighted): {weighted_log_loss(y_val, val_prior, w_val):.5f}")


# ── B) Logistic Regression (sweep over C) ──────────────────────────────
print("\n" + "=" * 60)
print("B) Logistic Regression — C sweep")
print("=" * 60)

best_lr, best_lr_ll, best_lr_C = None, 1e9, None
for C_val in [0.01, 0.1, 0.3, 1.0, 3.0]:
    pipe = Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(
            C=C_val, max_iter=500, solver="lbfgs",
            random_state=RANDOM_SEED, n_jobs=-1,
        )),
    ])
    pipe.fit(X_train, y_train, clf__sample_weight=w_train)
    vp = pipe.predict_proba(X_val)[:, 1]
    ll = weighted_log_loss(y_val, vp, w_val)
    print(f"  C={C_val:<5}  val logloss(w)={ll:.5f}")
    if ll < best_lr_ll:
        best_lr, best_lr_ll, best_lr_C = pipe, ll, C_val

print(f"  Best C={best_lr_C}")
models_registry["logistic_regression"] = {
    "model": best_lr,
    "val_pred": best_lr.predict_proba(X_val)[:, 1],
    "test_pred": best_lr.predict_proba(X_test)[:, 1],
}


# ── C) HistGradientBoosting ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("C) HistGradientBoostingClassifier")
print("=" * 60)

# HGB natively handles categoricals — mark event_type_id
cat_mask = [c in CAT_COLS for c in X_train.columns]
hgb = HistGradientBoostingClassifier(
    max_iter=400,
    max_depth=6,
    learning_rate=0.05,
    min_samples_leaf=50,
    max_bins=255,
    categorical_features=cat_mask,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=RANDOM_SEED,
)
hgb.fit(X_train, y_train, sample_weight=w_train)
vp = hgb.predict_proba(X_val)[:, 1]
ll = weighted_log_loss(y_val, vp, w_val)
print(f"  n_iter used: {hgb.n_iter_}")
print(f"  Val logloss (weighted): {ll:.5f}")
models_registry["hist_gbt"] = {
    "model": hgb,
    "val_pred": vp,
    "test_pred": hgb.predict_proba(X_test)[:, 1],
}

In [ ]:
# ── D) XGBoost ──────────────────────────────────────────────────────────
if RUN_XGB:
    print("=" * 60)
    print("D) XGBoost — small grid")
    print("=" * 60)

    # XGBoost needs a plain numeric matrix — use the preprocessor
    X_tr_xgb = preprocessor.fit_transform(X_train)
    X_va_xgb = preprocessor.transform(X_val)
    X_te_xgb = preprocessor.transform(X_test)

    best_xgb, best_xgb_ll = None, 1e9
    configs = [
        {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 600},
        {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 600},
        {"max_depth": 6, "learning_rate": 0.03, "n_estimators": 1000},
    ]
    for cfg in configs:
        xgb_clf = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            early_stopping_rounds=20,
            **cfg,
        )
        xgb_clf.fit(
            X_tr_xgb, y_train,
            sample_weight=w_train,
            eval_set=[(X_va_xgb, y_val)],
            sample_weight_eval_set=[w_val],
            verbose=False,
        )
        vp = xgb_clf.predict_proba(X_va_xgb)[:, 1]
        ll = weighted_log_loss(y_val, vp, w_val)
        print(f"  {cfg}  →  val logloss(w)={ll:.5f}  (best_iter={xgb_clf.best_iteration})")
        if ll < best_xgb_ll:
            best_xgb, best_xgb_ll = xgb_clf, ll

    models_registry["xgboost"] = {
        "model": best_xgb,
        "val_pred": best_xgb.predict_proba(X_va_xgb)[:, 1],
        "test_pred": best_xgb.predict_proba(X_te_xgb)[:, 1],
    }
else:
    print("Skipping XGBoost (not installed)")


# ── E) MLP Classifier ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("E) MLPClassifier (small neural net)")
print("=" * 60)

mlp_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        batch_size=2048,
        random_state=RANDOM_SEED,
    )),
])
# MLPClassifier doesn't support sample_weight in .fit() — train unweighted
mlp_pipe.fit(X_train, y_train)
vp = mlp_pipe.predict_proba(X_val)[:, 1]
ll = weighted_log_loss(y_val, vp, w_val)
print(f"  Val logloss (weighted): {ll:.5f}")
models_registry["mlp"] = {
    "model": mlp_pipe,
    "val_pred": vp,
    "test_pred": mlp_pipe.predict_proba(X_test)[:, 1],
}

print(f"\nModels trained: {list(models_registry.keys())}")

## 7. Evaluation (Weighted + Unweighted)

In [ ]:
# ─── 7. Evaluation ──────────────────────────────────────────────────────────

rows = []
for name, reg in models_registry.items():
    val_m  = compute_all_metrics(y_val,  reg["val_pred"],  w_val,  prefix="val")
    test_m = compute_all_metrics(y_test, reg["test_pred"], w_test, prefix="test")
    row = {"model": name, **val_m, **test_m}
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index("model")

# Pretty-print selected columns
display_cols = [
    "val_w_logloss", "val_w_brier", "val_w_ece", "val_w_auc",
    "test_w_logloss", "test_w_brier", "test_w_ece", "test_w_auc",
]
print("Evaluation Summary (weighted metrics):\n")
print(metrics_df[display_cols].round(5).to_string())

# ── Select best model by VAL weighted logloss ───────────────────────────
best_name = metrics_df["val_w_logloss"].idxmin()
print(f"\nBest model (by val weighted logloss): {best_name}")
print(f"  val  logloss(w) = {metrics_df.loc[best_name, 'val_w_logloss']:.5f}")
print(f"  test logloss(w) = {metrics_df.loc[best_name, 'test_w_logloss']:.5f}")

## 8. Calibration of Best Model

In [ ]:
# ─── 8. Calibration ─────────────────────────────────────────────────────────
#
# Fit an IsotonicRegression calibrator on the VALIDATION predictions
# of the best model, then apply to test predictions.

best_reg   = models_registry[best_name]
val_raw    = best_reg["val_pred"]
test_raw   = best_reg["test_pred"]

# ── Isotonic calibration (fitted on val) ────────────────────────────────
calibrator = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
calibrator.fit(val_raw, y_val, sample_weight=w_val)

val_cal  = calibrator.predict(val_raw)
test_cal = calibrator.predict(test_raw)

# ── Metrics before / after calibration ──────────────────────────────────
print("Test set — raw vs calibrated:\n")
raw_m = compute_all_metrics(y_test, test_raw, w_test, prefix="test_raw")
cal_m = compute_all_metrics(y_test, test_cal, w_test, prefix="test_cal")
for key in ["_w_logloss", "_w_brier", "_w_ece", "_w_auc"]:
    r = raw_m[f"test_raw{key}"]
    c = cal_m[f"test_cal{key}"]
    delta = c - r
    arrow = "^" if delta > 0 else "v"
    print(f"  {key[3:]:>10s}:  raw={r:.5f}   cal={c:.5f}   ({arrow} {abs(delta):.5f})")

# ── Reliability plots ───────────────────────────────────────────────────
plot_reliability_diagram(
    y_test, test_raw, w_test,
    title=f"Test — {best_name} (raw)",
    save_path=str(out_dir / "reliability_raw.png"),
)
plot_reliability_diagram(
    y_test, test_cal, w_test,
    title=f"Test — {best_name} (calibrated)",
    save_path=str(out_dir / "reliability_calibrated.png"),
)
print("\nCalibration complete")

## 9. Save Artifacts

In [ ]:
# ─── 9. Save Artifacts ──────────────────────────────────────────────────────

# ── 9a. Model artifact ──────────────────────────────────────────────────
best_model = best_reg["model"]
model_path = out_dir / "best_model.joblib"

if best_name == "xgboost" and best_model is not None:
    # Save native XGBoost JSON for portability + joblib wrapper
    xgb_json_path = out_dir / "best_model_xgb.json"
    best_model.save_model(str(xgb_json_path))
    print(f"  XGBoost JSON → {xgb_json_path}")
    # Also save the preprocessor that XGBoost needs
    joblib.dump(preprocessor, out_dir / "xgb_preprocessor.joblib")
    print(f"  XGBoost preprocessor → {out_dir / 'xgb_preprocessor.joblib'}")

if best_model is not None:
    joblib.dump(best_model, model_path)
    print(f"  Model artifact → {model_path}")
else:
    print("  Best model is the prior baseline — no artifact to save.")

# ── 9b. Calibrator ──────────────────────────────────────────────────────
cal_path = out_dir / "calibrator.joblib"
joblib.dump(calibrator, cal_path)
print(f"  Calibrator → {cal_path}")

# ── 9c. Metrics summary JSON ────────────────────────────────────────────
summary = {
    "best_model": best_name,
    "timestamp": datetime.now().isoformat(),
    "data_path": DATA_PATH,
    "random_seed": RANDOM_SEED,
    "train_rows": len(df_train),
    "val_rows": len(df_val),
    "test_rows": len(df_test),
    "feature_columns": list(X_train.columns),
    "numeric_features": NUMERIC_COLS,
    "categorical_features": CAT_COLS,
    "val_metrics_raw": {k: round(v, 6) for k, v in
                        compute_all_metrics(y_val, val_raw, w_val, "val").items()},
    "test_metrics_raw": {k: round(v, 6) for k, v in
                         compute_all_metrics(y_test, test_raw, w_test, "test").items()},
    "test_metrics_calibrated": {k: round(v, 6) for k, v in
                                compute_all_metrics(y_test, test_cal, w_test, "test_cal").items()},
    "all_models_val_logloss_w": {
        name: round(float(metrics_df.loc[name, "val_w_logloss"]), 6)
        for name in metrics_df.index
    },
}
json_path = out_dir / "metrics_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"  Metrics JSON → {json_path}")

# ── 9d. Test predictions CSV ────────────────────────────────────────────
pred_df = pd.DataFrame({
    "game_id":          df_test["game_id"].values,
    "play_index":       df_test["play_index"].values,
    "y_true":           y_test,
    "pred_raw":         test_raw,
    "pred_calibrated":  test_cal,
    "sample_weight":    w_test,
})
csv_path = out_dir / "test_predictions.csv"
pred_df.to_csv(csv_path, index=False)
print(f"  Test predictions CSV → {csv_path}  ({len(pred_df):,} rows)")

print("\nAll artifacts saved.")

## 10. Inference Utilities

In [ ]:
# ─── 10. Inference Utilities ────────────────────────────────────────────────

def load_artifacts(
    out_dir: str = OUT_DIR,
) -> Tuple[Any, IsotonicRegression, Dict]:
    """Load saved model, calibrator, and metadata from disk.

    Returns:
        model       — fitted sklearn/xgboost model (or Pipeline)
        calibrator  — fitted IsotonicRegression
        metadata    — dict from metrics_summary.json
    """
    od = Path(out_dir)
    metadata = json.loads((od / "metrics_summary.json").read_text())
    calibrator_ = joblib.load(od / "calibrator.joblib")

    model_name = metadata["best_model"]
    if model_name == "xgboost" and (od / "best_model_xgb.json").exists():
        from xgboost import XGBClassifier as XGB_
        m = XGB_()
        m.load_model(str(od / "best_model_xgb.json"))
    elif (od / "best_model.joblib").exists():
        m = joblib.load(od / "best_model.joblib")
    else:
        m = None   # prior baseline

    return m, calibrator_, metadata


def predict_proba(df: pd.DataFrame, model: Any = None) -> np.ndarray:
    """Produce raw P(home_win) from a DataFrame of snapshot features.

    If model is an XGBClassifier (not a Pipeline), the global `preprocessor`
    must already be fitted.
    """
    X, _, _ = make_features(df)
    if model is None:
        # Fallback to prior
        return df["prior_home_wp"].values.clip(1e-6, 1 - 1e-6)
    if hasattr(model, "predict_proba"):
        # XGBClassifier needs preprocessed input
        if isinstance(model, XGBClassifier):
            X = preprocessor.transform(X)
        return model.predict_proba(X)[:, 1]
    raise ValueError(f"Model type {type(model)} has no predict_proba")


def predict_proba_calibrated(
    df: pd.DataFrame,
    model: Any = None,
    calibrator_: Any = None,
) -> np.ndarray:
    """Produce calibrated P(home_win)."""
    raw = predict_proba(df, model)
    if calibrator_ is None:
        return raw
    return calibrator_.predict(raw)


# ── Demo: inference on 5 test rows ─────────────────────────────────────
print("Inference demo (5 test rows):\n")
demo_df = df_test.head(5).copy()
loaded_model, loaded_cal, loaded_meta = load_artifacts()
raw_p = predict_proba(demo_df, loaded_model)
cal_p = predict_proba_calibrated(demo_df, loaded_model, loaded_cal)

demo_out = pd.DataFrame({
    "game_id":    demo_df["game_id"].values,
    "play_index": demo_df["play_index"].values,
    "y_true":     demo_df["home_win_final"].values,
    "pred_raw":   raw_p,
    "pred_cal":   cal_p,
})
print(demo_out.to_string(index=False))
print(f"\nBest model: {loaded_meta['best_model']}")
print("Inference utilities ready.")